# Phase 3 — Core pipeline
**Goal:** build and test the four building blocks every experiment uses:

1. **Models**: the 6 classifiers with fixed settings
2. **Noise injection**: the 3 ways of making training labels wrong
3. **One experiment condition**: train a model 51 times and collect its predictions
4. **Bias–variance decomposition**: split the error into bias and variance

The code lives in **`src/core.py`**, not in this notebook. Phases 4, 5 and 6 all import the same file, so every experiment uses identical, tested code. This notebook explains each block and **tests** it before we trust it with days of computing.

Kernel: **Python (labelnoise)**.

In [1]:
import sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd

NB_DIR = Path.cwd()
BASE = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
sys.path.insert(0, str(BASE / "src"))
import core
from core import (load_dataset, make_model, reference_probs, symmetric_noise, one_sided_noise,
                  boundary_noise, run_condition, decompose, MODEL_NAMES, MODEL_FAMILY)

SEED = json.loads((BASE / "config.json").read_text())["SEED"]
DATA_DIR = BASE / "data"
X_tr, y_tr, X_te, y_te = load_dataset("wdbc", DATA_DIR)
print("wdbc  train", X_tr.shape, " test", X_te.shape, " class-1 share", round(y_tr.mean(), 3))

wdbc  train (398, 30)  test (171, 30)  class-1 share 0.372


## Block 1 — The models
We compare 6 models from three families:

| Model | Family | How it works | Expected behaviour under noise |
|---|---|---|---|
| Tree | single | One unpruned decision tree | Baseline: very unstable (high variance) |
| Bagging | parallel | 100 trees on bootstrap samples, majority vote | Voting should outvote wrong labels |
| RF | parallel | Bagging + a random subset of features at each split | Even more stable than Bagging |
| AdaBoost | sequential | 100 stumps; each round up-weights misclassified points | Keeps chasing wrong labels: fragile |
| GB | sequential | 100 small trees, each fits the previous errors (learning rate 0.1, depth 3) | Somewhere in between |
| XGB | sequential | Modern, regularised gradient boosting (depth 4) | The industry standard; how robust is it? |

**Key design rule: settings are frozen.** The same settings are used for every noise condition. If we re-tuned for each noise level, part of the change in error would come from re-tuning, not from noise. Freezing them also matches reality: a practitioner usually doesn't know how noisy their labels are, so they can't tune for it.

**Speed design:** each model uses 1 CPU thread, and we train **many rounds at the same time** (one per thread). On small datasets that is much faster than giving one model all 12 threads. That was why Bagging and XGB looked slow in Phase 0.

In [2]:
rows = []
for name in MODEL_NAMES:
    m = make_model(name, seed=0)
    t = time.perf_counter(); m.fit(X_tr, y_tr); fit_s = time.perf_counter() - t
    rows.append({"model": name, "family": MODEL_FAMILY[name], "class": type(m).__name__,
                 "clean test accuracy": round(m.score(X_te, y_te), 3), "fit seconds": round(fit_s, 3)})
pd.DataFrame(rows)

,model,family,class,clean test accuracy,fit seconds
0,Tree,single,DecisionTreeClassifier,0.901,0.007
1,Bagging,parallel,BaggingClassifier,0.977,0.269
2,RF,parallel,RandomForestClassifier,0.965,0.106
3,AdaBoost,sequential,AdaBoostClassifier,0.977,0.157
4,GB,sequential,GradientBoostingClassifier,0.965,0.232
5,XGB,sequential,XGBClassifier,0.982,2.108


## Block 2 — Noise injection
All three functions flip **exactly** a fixed number of labels, so "20 % noise" always means 20 %, not "about 20 % on average". The three types differ only in **which** labels get flipped:

| Function | Which labels are flipped | Survey term |
|---|---|---|
| `symmetric_noise` | Any label, uniformly at random | NCAR (completely at random) |
| `one_sided_noise` | Only class 1 → 0 (missed positives). `rho` = share **of the positives** | NAR (depends on the class) |
| `boundary_noise` | Mostly points the reference model finds hard (close to 50/50) | NNAR (depends on the example) |

**The reference model for boundary noise:** a logistic regression trained on the **clean** labels. For each training row it gives P(class 1). A point with P = 0.5 sits right on the decision boundary and gets the highest flip chance; a point with P = 0.99 almost never gets flipped. This copies how humans err: mostly on ambiguous cases.

The tests below check each function does exactly what it claims.

In [3]:
rng = np.random.default_rng(0)
rho = 0.20
p_ref = reference_probs(X_tr, y_tr, seed=SEED)
margin = np.abs(p_ref - 0.5)              # 0 = on the boundary, 0.5 = far from it

ys = symmetric_noise(y_tr, rho, rng)
yo = one_sided_noise(y_tr, rho, rng)
yb = boundary_noise(y_tr, rho, rng, p_ref)

def describe(name, y_noisy):
    flipped = y_noisy != y_tr
    return {"noise": name,
            "labels flipped": int(flipped.sum()),
            "flip rate (all rows)": round(flipped.mean(), 3),
            "flips 1->0": int((flipped & (y_tr == 1)).sum()),
            "flips 0->1": int((flipped & (y_tr == 0)).sum()),
            "avg margin of flipped": round(margin[flipped].mean(), 3),
            "class-1 share after": round(y_noisy.mean(), 3)}

print("training rows:", len(y_tr), "| positives:", int(y_tr.sum()),
      "| avg margin of all rows:", round(margin.mean(), 3))
pd.DataFrame([describe("symmetric", ys), describe("one_sided", yo), describe("boundary", yb)])

training rows: 398 | positives: 148 | avg margin of all rows: 0.468


,noise,labels flipped,flip rate (all rows),flips 1->0,flips 0->1,avg margin of flipped,class-1 share after
0,symmetric,80,0.201,38,42,0.469,0.382
1,one_sided,30,0.075,30,0,0.461,0.296
2,boundary,80,0.201,30,50,0.368,0.422


**How to read the table (and what the tests check):**
- **symmetric:** exactly 20 % of all rows flipped, in both directions. The flipped points have a normal average margin, because they were picked at random.
- **one_sided:** exactly 20 % of the *positives* flipped, only 1 → 0. The overall flip rate is lower, because positives are only 37 % of the rows. The class-1 share drops, so **one-sided noise changes the class balance**. Phase 5 records this.
- **boundary:** exactly 20 % of all rows flipped, but their **average margin is much smaller** than that of all rows. The flips really are concentrated near the boundary.

In [4]:
n, n_pos = len(y_tr), int(y_tr.sum())
assert (ys != y_tr).sum() == round(rho * n)
assert (yo != y_tr).sum() == round(rho * n_pos) and ((yo != y_tr) & (y_tr == 0)).sum() == 0
assert (yb != y_tr).sum() == round(rho * n)
assert margin[yb != y_tr].mean() < margin[ys != y_tr].mean()
print("All noise tests passed.")

All noise tests passed.


## Block 3 — One experiment condition (`run_condition`)
One **condition** = one (dataset, model, noise type, noise rate). For it we repeat **R = 51 rounds**:

1. Draw a **bootstrap sample** of the training data (random rows, with replacement). Each round sees a slightly different training set, as if we had collected the data again.
2. **Inject noise** into that sample's labels.
3. **Train** the model and **predict** the clean test set.

The predictions are stacked into a matrix **P** with 51 rows (rounds) and one column per test point, the same structure you built by hand in the Phase 1 learning guide.

**Why 51?** It is odd, so the majority vote can never tie, and it is large enough for stable estimates.

**Fair comparison ("common random numbers"):** round *r* always uses the same bootstrap sample and the same flipped labels, whichever model is trained. So when RF and AdaBoost are compared, they faced *exactly* the same data, and any difference comes from the models themselves.

In [5]:
t = time.perf_counter()
P, stats = run_condition(X_tr, y_tr, X_te, "RF", "symmetric", 0.20, R=51, seed=SEED)
print(f"P shape: {P.shape}  (rounds x test points)   time: {time.perf_counter() - t:.1f} s")
print("average realised flip rate:", round(stats["flip_rate"], 3))
print("first 3 rounds, first 12 test points:\n", P[:3, :12])

P shape: (51, 171)  (rounds x test points)   time: 2.8 s
average realised flip rate: 0.201
first 3 rounds, first 12 test points:
 [[0 0 0 0 0 0 1 0 1 0 0 1]
 [0 0 0 0 0 0 1 0 1 0 0 1]
 [0 0 0 0 0 0 1 0 1 0 0 1]]


## Block 4 — The bias–variance decomposition (`decompose`)
For each test point:
- **main prediction** = majority vote of the 51 models
- **bias** = 1 if the main prediction is wrong
- **variance** = share of the 51 models that disagree with the main prediction

Averaged over the test set, Domingos' identity must hold **exactly**:

  **error = bias + unbiased variance − biased variance**

**Test 1:** the worked example from the learning guide must give error 0.55, bias 0.50, unbiased variance 0.10, biased variance 0.05.
**Test 2:** the identity must hold on thousands of random prediction matrices. If it ever fails, the code has a bug.

In [6]:
P_example = np.array([[1, 1, 0, 0], [1, 1, 0, 1], [1, 0, 0, 0], [0, 1, 0, 0], [1, 1, 0, 0]])
d = decompose(P_example, np.array([1, 0, 1, 0]))
print({k: round(v, 4) for k, v in d.items()})
assert np.allclose([d["error"], d["bias"], d["var_unbiased"], d["var_biased"]], [0.55, 0.50, 0.10, 0.05])

rng = np.random.default_rng(1)
worst = max(abs(decompose(rng.integers(0, 2, (51, 200)), rng.integers(0, 2, 200))["identity_gap"])
            for _ in range(2000))
print("largest identity gap over 2,000 random matrices:", worst)
assert worst < 1e-12
print("All decomposition tests passed.")

{'error': 0.55, 'bias': 0.5, 'var_unbiased': 0.1, 'var_biased': 0.05, 'net_var': 0.05, 'identity_gap': 0.0}
largest identity gap over 2,000 random matrices: 1.1102230246251565e-16
All decomposition tests passed.


## Putting it together: a first real result on wdbc
All 6 models, clean labels vs **20 % symmetric noise**, 51 rounds each. This is a small preview of Phase 5.

**Columns:**
- **error**: share of wrong test predictions
- **bias / net_var**: the two parts of the error (net_var = unbiased − biased variance)
- **Δ error, Δ bias, Δ net_var**: the change caused by noise. This is what the paper is about.

In [7]:
results = []
t0 = time.perf_counter()
for name in MODEL_NAMES:
    for rho in (0.0, 0.20):
        P, stats = run_condition(X_tr, y_tr, X_te, name, "symmetric", rho, R=51, seed=SEED)
        results.append({"dataset": "wdbc", "model": name, "family": MODEL_FAMILY[name],
                        "noise_type": "symmetric", "rho": rho, **decompose(P, y_te), **stats})
print(f"12 conditions x 51 rounds in {time.perf_counter() - t0:.0f} s")

df = pd.DataFrame(results)
clean = df[df.rho == 0].set_index("model")
noisy = df[df.rho == 0.20].set_index("model")
summary = pd.DataFrame({
    "family": clean["family"],
    "error clean": clean["error"], "error 20%": noisy["error"],
    "Δ error": noisy["error"] - clean["error"],
    "Δ bias": noisy["bias"] - clean["bias"],
    "Δ net_var": noisy["net_var"] - clean["net_var"],
}).round(3).loc[MODEL_NAMES]
summary

12 conditions x 51 rounds in 16 s


,family,error clean,error 20%,Δ error,Δ bias,Δ net_var
model,,,,,,
Tree,single,0.078,0.223,0.145,0.006,0.139
Bagging,parallel,0.049,0.082,0.032,0.006,0.027
RF,parallel,0.039,0.060,0.021,0.012,0.009
AdaBoost,sequential,0.028,0.104,0.076,0.035,0.041
GB,sequential,0.044,0.098,0.054,0.012,0.043
XGB,sequential,0.033,0.090,0.057,0.023,0.034


**How to read this preview** (one small dataset, one noise type, so it shows *how the method works*, not conclusions):
- **Δ error** says how much each model got worse.
- **Δ bias + Δ net_var = Δ error**, so the last two columns split the damage into its two kinds.
- Compare the **parallel** rows (Bagging, RF) with the **sequential** rows (AdaBoost, GB, XGB). In Phase 5 we repeat this for 15 datasets × 3 noise types × 4 rates and test whether the pattern holds.

In [8]:
assert df["identity_gap"].abs().max() < 1e-12
out = BASE / "results" / "raw" / "_phase3_demo.csv"
df.to_csv(out, index=False)
print("Identity holds on every row. Saved:", out)

Identity holds on every row. Saved: /home/gravtion/VS Code/College/ML Research Paper/results/raw/_phase3_demo.csv


## Phase 3 checklist
- [ ] Block 1: all 6 models train and score on wdbc
- [ ] Block 2: all noise tests passed
- [ ] Block 3: `P` has shape (51, n_test)
- [ ] Block 4: worked example and identity tests passed
- [ ] Preview table produced and saved to `results/raw/_phase3_demo.csv`

**Next: Phase 4 (pilot)** runs a small grid on 2 datasets, checks the six sanity conditions and measures run time, before the long Phase 5 runs.